# Data Fetching and Preprocessing
This cell handles data fetching, indicator calculation, and data saving

In [18]:
from tvDatafeed import TvDatafeed, Interval
import pandas as pd
import numpy as np
from talipp.indicators import MACD, RSI, EMA
from talipp.indicator_util import composite_to_lists
import logging
import os
import pandas_ta as ta

logger = logging.getLogger(__name__)
username = 'Chiqo-254'
password = 'ChiqoMehum8844'

tv = TvDatafeed(username,password)

h1_data = tv.get_hist(symbol='EURUSD',exchange='FX_IDC',interval=Interval.in_1_hour,n_bars=5000)
m5_data = tv.get_hist(symbol='EURUSD',exchange='FX_IDC',interval=Interval.in_5_minute,n_bars=8640)

#create data directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')
if h1_data is not None:
    h1_data.to_csv('data/h1_data.csv')
if m5_data is not None: 
    m5_data.to_csv('data/m5_data.csv')
    
    


ERROR:tvDatafeed.main:error while signin


In [19]:
import os
import numpy as np
import pandas_ta as ta
import pandas as pd

# Load the data
h1_data = pd.read_csv('data/h1_data.csv')
m5_data = pd.read_csv('data/m5_data.csv')

# Calculate H1 trend bias
def calculate_h1_trend_bias(h1_data):
    # Trend Bias: 8-period EMA on 1-hour chart
    h1_data['ema_8'] = ta.ema(h1_data['close'], length=8)
    h1_data['trend_bias'] = np.where(
        h1_data['close'] > h1_data['ema_8'], 
        'bullish', 
        'bearish'
    )
    return h1_data['trend_bias'].iloc[-1]

# Calculate technical indicators
def calculate_and_save_indicators(file_path, is_m5=False, h1_trend_bias=None):
    try:
        df = pd.read_csv(file_path)
        if 'close' not in df.columns:
            raise ValueError(f"'close' column not found in {file_path}")
        
        if is_m5:
            print("Calculating 5M indicators...")
            # Add H1 trend bias to M5 data if applicable
            if h1_trend_bias is not None:
                df['trend_bias'] = h1_trend_bias
        else:
            print("Calculating 1H indicators...")
            
        # EMA
        df['ema_5'] = ta.ema(df['close'], length=5)
        
        # EMA crossover signals
        df['price_crossed_above_ema'] = np.where(
            (df['close'] > df['ema_5']) & (df['close'].shift(1) <= df['ema_5'].shift(1)),
            True,
            False
        )
        df['price_crossed_below_ema'] = np.where(
            (df['close'] < df['ema_5']) & (df['close'].shift(1) >= df['ema_5'].shift(1)),
            True,
            False
        )
        
        # RSI indicator for divergence
        df['rsi_9'] = ta.rsi(df['close'], length=9)
        
        # MACD for exit trades
        macd = ta.macd(df['close'], fast=8, slow=17, signal=9)
        df = df.join(macd)
        
        # Identify MACD crossovers
        df['macd_cross_below'] = np.where(
            (df['MACD_8_17_9'] < df['MACDs_8_17_9']) & 
            (df['MACD_8_17_9'].shift(1) >= df['MACDs_8_17_9'].shift(1)),
            True,
            False
        )
        df['macd_cross_above'] = np.where(
            (df['MACD_8_17_9'] > df['MACDs_8_17_9']) & 
            (df['MACD_8_17_9'].shift(1) <= df['MACDs_8_17_9'].shift(1)),
            True,
            False
        )
        
        # Save back to the same file
        df.to_csv(file_path, index=False)
        print(f"Indicators calculated and saved for {file_path}")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# First calculate H1 data and get trend bias
calculate_and_save_indicators('data/h1_data.csv')
h1_trend_bias = calculate_h1_trend_bias(pd.read_csv('data/h1_data.csv'))

# Then calculate M5 data with H1 trend bias
calculate_and_save_indicators('data/m5_data.csv', is_m5=True, h1_trend_bias=h1_trend_bias)


Calculating 1H indicators...
Indicators calculated and saved for data/h1_data.csv
Calculating 5M indicators...
Indicators calculated and saved for data/h1_data.csv
Calculating 5M indicators...
Indicators calculated and saved for data/m5_data.csv
Indicators calculated and saved for data/m5_data.csv


In [38]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots  # Add this import

h1_data = pd.read_csv('data/h1_data.csv')
m5_data = pd.read_csv('data/m5_data.csv')

# Recalculate trend bias for h1_data
h1_data['ema_8'] = ta.ema(h1_data['close'], length=8)
h1_data['trend_bias'] = np.where(
    h1_data['close'] > h1_data['ema_8'], 
    'bullish', 
    'bearish'
)

# Now create signals DataFrame with updated data
signals_df = pd.DataFrame({
    'datetime': m5_data.index,
    'close': m5_data['close'],
    'ema_5': m5_data['ema_5'],
    'rsi_9': m5_data['rsi_9'],
    'macd': m5_data['MACD_8_17_9'],
    'macd_signal': m5_data['MACDs_8_17_9'],
    'price_crossed_above_ema': m5_data['price_crossed_above_ema'],
    'price_crossed_below_ema': m5_data['price_crossed_below_ema'],
    'macd_cross_above': m5_data['macd_cross_above'],
    'macd_cross_below': m5_data['macd_cross_below']
})

# Add hourly trend bias by resampling to 5-minute timeframe
h1_data['datetime'] = pd.to_datetime(h1_data['datetime'])
m5_data['datetime'] = pd.to_datetime(m5_data['datetime'])
h1_data.set_index('datetime', inplace=True)
m5_data.set_index('datetime', inplace=True)
signals_df['trend_bias'] = h1_data['trend_bias'].reindex(m5_data.index, method='ffill')

# --- NEW: FILTER ENTRIES BY TREND BIAS ---
# Long entries: Only when 1H trend is bullish
signals_df['price_crossed_above_ema'] = (
    m5_data['price_crossed_above_ema']  # Original crossover condition
    & (signals_df['trend_bias'] == 'bullish')  # Trend filter
)

# Short entries: Only when 1H trend is bearish
signals_df['price_crossed_below_ema'] = (
    m5_data['price_crossed_below_ema']  # Original crossover condition
    & (signals_df['trend_bias'] == 'bearish')  # Trend filter
)

# First define our analysis function
def analyze_strategy_performance(m5_data, signals_df, stop_loss_pips=5, risk_per_trade=0.01):
    """
    Analyze trading strategy performance metrics with stop loss
    """
    # Initialize trade tracking variables
    trades = []
    position = None
    entry_price = 0
    entry_time = None
    pnl = []
    stop_loss_price = 0
    pip_value = 0.0001
    
    # Iterate through the data
    for i in range(len(m5_data)):
        current_price = m5_data['close'].iloc[i]
        current_time = m5_data.index[i]
        current_trend = m5_data['trend_bias'].iloc[i]
        
        # Check for stop loss if in position
        if position == 'long' and current_price <= stop_loss_price:
            pnl_pips = (stop_loss_price - entry_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': stop_loss_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'stop_loss'
            })
            position = None
            pnl.append(pnl_pips)
            continue
            
        elif position == 'short' and current_price >= stop_loss_price:
            pnl_pips = (entry_price - stop_loss_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': stop_loss_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'stop_loss'
            })
            position = None
            pnl.append(pnl_pips)
            continue
        
        # Check for entry signals - Only enter if aligned with trend bias
        if position is None:
            # Long entries only during bullish trend
            if (current_trend == 'bullish' and 
                m5_data['price_crossed_above_ema'].iloc[i]):
                position = 'long'
                entry_price = current_price
                entry_time = current_time
                stop_loss_price = entry_price - (stop_loss_pips * pip_value)
                
            # Short entries only during bearish trend
            elif (current_trend == 'bearish' and 
                  m5_data['price_crossed_below_ema'].iloc[i]):
                position = 'short'
                entry_price = current_price
                entry_time = current_time
                stop_loss_price = entry_price + (stop_loss_pips * pip_value)
        
        # Check for exit signals
        elif position == 'long' and m5_data['macd_cross_below'].iloc[i]:
            pnl_pips = (current_price - entry_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': current_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'signal'
            })
            position = None
            pnl.append(pnl_pips)
            
        elif position == 'short' and m5_data['macd_cross_above'].iloc[i]:
            pnl_pips = (entry_price - current_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': current_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'signal'
            })
            position = None
            pnl.append(pnl_pips)
    
    # Convert trades list to DataFrame
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) == 0:
        print("No trades executed in the given period")
        return None
        
    # Calculate performance metrics
    total_trades = len(trades_df)
    winning_trades = len(trades_df[trades_df['pnl_pips'] > 0])
    losing_trades = len(trades_df[trades_df['pnl_pips'] <= 0])
    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
    
    avg_win = trades_df[trades_df['pnl_pips'] > 0]['pnl_pips'].mean() if winning_trades > 0 else 0
    avg_loss = abs(trades_df[trades_df['pnl_pips'] <= 0]['pnl_pips'].mean()) if losing_trades > 0 else 0
    profit_factor = (avg_win * winning_trades) / (avg_loss * losing_trades) if losing_trades > 0 and avg_loss > 0 else 0
    
    # Calculate running equity curve and drawdown
    cumulative_pnl = np.cumsum(trades_df['pnl_pips'])
    running_max = np.maximum.accumulate(cumulative_pnl)
    drawdown = running_max - cumulative_pnl
    max_drawdown = max(drawdown) if len(drawdown) > 0 else 0
    
    # Calculate average trade duration
    avg_duration = trades_df['duration'].mean()
    
    # Print basic metrics
    print("\n=== Strategy Performance Metrics ===")
    print(f"Total Trades: {total_trades}")
    print(f"Winning Trades: {winning_trades}")
    print(f"Losing Trades: {losing_trades}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Average Win: {avg_win:.2f} pips")
    print(f"Average Loss: {avg_loss:.2f} pips")
    print(f"Profit Factor: {profit_factor:.2f}")
    print(f"Maximum Drawdown: {max_drawdown:.2f} pips")
    print(f"Average Trade Duration: {avg_duration:.2f} minutes")
    
    # Add stop loss analysis
    stop_loss_exits = len(trades_df[trades_df['exit_type'] == 'stop_loss'])
    signal_exits = len(trades_df[trades_df['exit_type'] == 'signal'])
    print(f"\nExit Analysis:")
    print(f"Stop Loss Exits: {stop_loss_exits} ({(stop_loss_exits/total_trades*100):.2f}%)")
    print(f"Signal Exits: {signal_exits} ({(signal_exits/total_trades*100):.2f}%)")
    
    # Calculate metrics by exit type
    sl_pnl = trades_df[trades_df['exit_type'] == 'stop_loss']['pnl_pips'].mean()
    signal_pnl = trades_df[trades_df['exit_type'] == 'signal']['pnl_pips'].mean()
    print(f"Average Stop Loss Trade: {sl_pnl:.2f} pips")
    print(f"Average Signal Exit Trade: {signal_pnl:.2f} pips")
    
    # Visualize equity curve and drawdown
    fig = make_subplots(rows=2, cols=1, 
                       subplot_titles=('Cumulative Profit/Loss', 'Drawdown'),
                       vertical_spacing=0.12)
    
    # Equity curve
    fig.add_trace(
        go.Scatter(y=cumulative_pnl, name="Equity Curve",
                  line=dict(color='blue')),
        row=1, col=1
    )
    
    # Drawdown
    fig.add_trace(
        go.Scatter(y=-drawdown, name="Drawdown",
                  fill='tonexty', line=dict(color='red')),
        row=2, col=1
    )
    
    fig.update_layout(
        height=800,
        title_text="Strategy Performance",
        showlegend=True
    )
    
    fig.show()
    
    return trades_df

# Function to visualize the indicators for a specific time period (optional)
def visualize_indicators(h1_sample, m5_sample, signals_sample, trades_df=None, period=50):  # Added trades_df parameter
    """
    Visualize the indicators for analysis using Plotly for interactive charts
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    
    # Create figure with secondary y-axis
    fig = make_subplots(rows=2, cols=1, 
                       subplot_titles=('1H Chart - Trend Bias', '5M Chart - Entry & Exit Points'),
                       vertical_spacing=0.1,
                       row_heights=[0.5, 0.5])
    
    # Add 1H Chart traces
    fig.add_trace(
        go.Scatter(x=h1_sample.index[-period:], 
                  y=h1_sample['close'][-period:],
                  name="Close Price (1H)",
                  line=dict(color='blue')),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=h1_sample.index[-period:], 
                  y=h1_sample['ema_8'][-period:],
                  name="EMA(8)",
                  line=dict(color='orange')),
        row=1, col=1
    )
    
    # Add 5M Chart traces
    fig.add_trace(
        go.Scatter(x=m5_sample.index[-period*12:],
                  y=m5_sample['close'][-period*12:],
                  name="Close Price (5M)",
                  line=dict(color='blue')),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=m5_sample.index[-period*12:],
                  y=m5_sample['ema_5'][-period*12:],
                  name="EMA(5)",
                  line=dict(color='orange')),
        row=2, col=1
    )
    
    # Add entry signals using signals_df
    entries_long = signals_sample[-period*12:][signals_sample['price_crossed_above_ema'][-period*12:]]
    entries_short = signals_sample[-period*12:][signals_sample['price_crossed_below_ema'][-period*12:]]
    
    fig.add_trace(
        go.Scatter(x=entries_long.index,
                  y=m5_sample['close'][entries_long.index],  # Get prices from m5_data
                  mode='markers',
                  name='Long Entry',
                  marker=dict(color='green', size=10, symbol='triangle-up')),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=entries_short.index,
                  y=m5_sample['close'][entries_short.index],  # Get prices from m5_data
                  mode='markers',
                  name='Short Entry',
                  marker=dict(color='red', size=10, symbol='triangle-down')),
        row=2, col=1
    )
    
    # Add actual trade entry points if trades_df is provided
    if trades_df is not None:
        # Filter trades within the visualization period
        recent_trades = trades_df[trades_df['entry_time'] >= m5_sample.index[-period*12:][0]]
        
        # Plot long trade entries
        long_entries = recent_trades[recent_trades['position'] == 'long']
        if not long_entries.empty:
            fig.add_trace(
                go.Scatter(x=long_entries['entry_time'],
                          y=long_entries['entry_price'],
                          mode='markers',
                          name='Executed Long Entry',
                          marker=dict(color='green', size=12, symbol='triangle-up')),
                row=2, col=1
            )
        
        # Plot short trade entries
        short_entries = recent_trades[recent_trades['position'] == 'short']
        if not short_entries.empty:
            fig.add_trace(
                go.Scatter(x=short_entries['entry_time'],
                          y=short_entries['entry_price'],
                          mode='markers',
                          name='Executed Short Entry',
                          marker=dict(color='red', size=12, symbol='triangle-down')),
                row=2, col=1
            )
    
    # Add exit signals
    exits_long = m5_sample[-period*12:][m5_sample['macd_cross_below'][-period*12:]]
    exits_short = m5_sample[-period*12:][m5_sample['macd_cross_above'][-period*12:]]
    
    fig.add_trace(
        go.Scatter(x=exits_long.index,
                  y=exits_long['close'],
                  mode='markers',
                  name='Long Exit',
                  marker=dict(color='orange', size=10, symbol='x')),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(x=exits_short.index,
                  y=exits_short['close'],
                  mode='markers',
                  name='Short Exit',
                  marker=dict(color='purple', size=10, symbol='x')),
        row=2, col=1
    )
    
    # Update layout
    fig.update_layout(
        height=800,
        showlegend=True,
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="left",
            x=1.05
        ),
        xaxis_rangeslider_visible=False,
        xaxis2_rangeslider_visible=False
    )
    
    # Add grid and finalize layout
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
    
    # Show the plot
    fig.show()

# Now run analysis first
trades_df = analyze_strategy_performance(m5_data, signals_df, stop_loss_pips=5)

# Then run visualization with the trades_df
visualize_indicators(h1_data, m5_data, signals_df, trades_df)


=== Strategy Performance Metrics ===
Total Trades: 422
Winning Trades: 133
Losing Trades: 289
Win Rate: 31.52%
Average Win: 12.83 pips
Average Loss: 4.13 pips
Profit Factor: 1.43
Maximum Drawdown: 109.70 pips
Average Trade Duration: 58.21 minutes

Exit Analysis:
Stop Loss Exits: 195 (46.21%)
Signal Exits: 227 (53.79%)
Average Stop Loss Trade: -5.00 pips
Average Signal Exit Trade: 6.55 pips


/tmp/ipykernel_312352/1214426360.py:286: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipykernel_312352/1214426360.py:295: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



In [ ]:
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def analyze_strategy_performance(m5_data, signals_df, stop_loss_pips=10, risk_per_trade=0.01):
    """
    Analyze trading strategy performance metrics with stop loss
    Only enter trades in the direction of the 1-hour trend bias
    """
    # Initialize trade tracking variables
    trades = []
    position = None
    entry_price = 0
    entry_time = None
    pnl = []
    stop_loss_price = 0
    pip_value = 0.0001
    
    # Iterate through the data
    for i in range(len(m5_data)):
        current_price = m5_data['close'].iloc[i]
        current_time = m5_data.index[i]
        current_trend = m5_data['trend_bias'].iloc[i]
        
        # Check for stop loss if in position
        if position == 'long' and current_price <= stop_loss_price:
            pnl_pips = (stop_loss_price - entry_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': stop_loss_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'stop_loss'
            })
            position = None
            pnl.append(pnl_pips)
            continue
            
        elif position == 'short' and current_price >= stop_loss_price:
            pnl_pips = (entry_price - stop_loss_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': stop_loss_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'stop_loss'
            })
            position = None
            pnl.append(pnl_pips)
            continue
        
        # Check for entry signals - Only enter if aligned with trend bias
        if position is None:
            # Long entries only during bullish trend
            if (current_trend == 'bullish' and 
                m5_data['price_crossed_above_ema'].iloc[i]):
                position = 'long'
                entry_price = current_price
                entry_time = current_time
                stop_loss_price = entry_price - (stop_loss_pips * pip_value)
                
            # Short entries only during bearish trend
            elif (current_trend == 'bearish' and 
                  m5_data['price_crossed_below_ema'].iloc[i]):
                position = 'short'
                entry_price = current_price
                entry_time = current_time
                stop_loss_price = entry_price + (stop_loss_pips * pip_value)
        
        # Check for exit signals
        elif position == 'long' and m5_data['macd_cross_below'].iloc[i]:
            pnl_pips = (current_price - entry_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': current_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'signal'
            })
            position = None
            pnl.append(pnl_pips)
            
        elif position == 'short' and m5_data['macd_cross_above'].iloc[i]:
            pnl_pips = (entry_price - current_price) / pip_value
            trades.append({
                'entry_time': entry_time,
                'exit_time': current_time,
                'entry_price': entry_price,
                'exit_price': current_price,
                'position': position,
                'pnl_pips': pnl_pips,
                'duration': (current_time - entry_time).total_seconds() / 60,
                'exit_type': 'signal'
            })
            position = None
            pnl.append(pnl_pips)
    
    # Convert trades list to DataFrame
    trades_df = pd.DataFrame(trades)
    
    if len(trades_df) == 0:
        print("No trades executed in the given period")
        return None
        
    # Calculate performance metrics
    total_trades = len(trades_df)
    winning_trades = len(trades_df[trades_df['pnl_pips'] > 0])
    losing_trades = len(trades_df[trades_df['pnl_pips'] <= 0])
    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
    
    avg_win = trades_df[trades_df['pnl_pips'] > 0]['pnl_pips'].mean() if winning_trades > 0 else 0
    avg_loss = abs(trades_df[trades_df['pnl_pips'] <= 0]['pnl_pips'].mean()) if losing_trades > 0 else 0
    profit_factor = (avg_win * winning_trades) / (avg_loss * losing_trades) if losing_trades > 0 and avg_loss > 0 else 0
    
    # Calculate running equity curve and drawdown
    cumulative_pnl = np.cumsum(trades_df['pnl_pips'])
    running_max = np.maximum.accumulate(cumulative_pnl)
    drawdown = running_max - cumulative_pnl
    max_drawdown = max(drawdown) if len(drawdown) > 0 else 0
    
    # Calculate average trade duration
    avg_duration = trades_df['duration'].mean()
    
    # Print basic metrics
    print("\n=== Strategy Performance Metrics ===")
    print(f"Total Trades: {total_trades}")
    print(f"Winning Trades: {winning_trades}")
    print(f"Losing Trades: {losing_trades}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Average Win: {avg_win:.2f} pips")
    print(f"Average Loss: {avg_loss:.2f} pips")
    print(f"Profit Factor: {profit_factor:.2f}")
    print(f"Maximum Drawdown: {max_drawdown:.2f} pips")
    print(f"Average Trade Duration: {avg_duration:.2f} minutes")
    
    # Add stop loss analysis
    stop_loss_exits = len(trades_df[trades_df['exit_type'] == 'stop_loss'])
    signal_exits = len(trades_df[trades_df['exit_type'] == 'signal'])
    print(f"\nExit Analysis:")
    print(f"Stop Loss Exits: {stop_loss_exits} ({(stop_loss_exits/total_trades*100):.2f}%)")
    print(f"Signal Exits: {signal_exits} ({(signal_exits/total_trades*100):.2f}%)")
    
    # Calculate metrics by exit type
    sl_pnl = trades_df[trades_df['exit_type'] == 'stop_loss']['pnl_pips'].mean()
    signal_pnl = trades_df[trades_df['exit_type'] == 'signal']['pnl_pips'].mean()
    print(f"Average Stop Loss Trade: {sl_pnl:.2f} pips")
    print(f"Average Signal Exit Trade: {signal_pnl:.2f} pips")
    
    # Visualize equity curve and drawdown
    fig = make_subplots(rows=2, cols=1, 
                       subplot_titles=('Cumulative Profit/Loss', 'Drawdown'),
                       vertical_spacing=0.12)
    
    # Equity curve
    fig.add_trace(
        go.Scatter(y=cumulative_pnl, name="Equity Curve",
                  line=dict(color='blue')),
        row=1, col=1
    )
    
    # Drawdown
    fig.add_trace(
        go.Scatter(y=-drawdown, name="Drawdown",
                  fill='tonexty', line=dict(color='red')),
        row=2, col=1
    )
    
    fig.update_layout(
        height=800,
        title_text="Strategy Performance",
        showlegend=True
    )
    
    fig.show()
    
    return trades_df

# Run the analysis with trend-filtered entries
trades_df = analyze_strategy_performance(m5_data, signals_df, stop_loss_pips=10)

if trades_df is not None:
    # Add trend bias to trade analysis
    print("\n=== Trend Analysis ===")
    bullish_trades = len(trades_df[trades_df['position'] == 'long'])
    bearish_trades = len(trades_df[trades_df['position'] == 'short'])
    print(f"Bullish Trades: {bullish_trades}")
    print(f"Bearish Trades: {bearish_trades}")
    
    # Calculate performance by trend
    long_pnl = trades_df[trades_df['position'] == 'long']['pnl_pips'].mean()
    short_pnl = trades_df[trades_df['position'] == 'short']['pnl_pips'].mean()
    print(f"Average Long Trade: {long_pnl:.2f} pips")
    print(f"Average Short Trade: {short_pnl:.2f} pips")
    
    print("\n=== Recent Trades ===")
    pd.set_option('display.max_columns', None)
    print(trades_df.tail().to_string())


=== Strategy Performance Metrics ===
Total Trades: 319
Winning Trades: 127
Losing Trades: 192
Win Rate: 39.81%
Average Win: 13.06 pips
Average Loss: 7.22 pips
Profit Factor: 1.20
Maximum Drawdown: 143.80 pips
Average Trade Duration: 92.66 minutes

Exit Analysis:
Stop Loss Exits: 39 (12.23%)
Signal Exits: 280 (87.77%)
Average Stop Loss Trade: -15.00 pips
Average Signal Exit Trade: 3.06 pips



=== Trend Analysis ===
Bullish Trades: 0
Bearish Trades: 319
Average Long Trade: nan pips
Average Short Trade: 0.85 pips

=== Recent Trades ===
             entry_time           exit_time  entry_price  exit_price position  pnl_pips  duration exit_type
314 2025-05-16 04:00:00 2025-05-16 05:10:00      1.12029     1.12079    short      -5.0      70.0    signal
315 2025-05-16 05:20:00 2025-05-16 06:30:00      1.12039     1.12046    short      -0.7      70.0    signal
316 2025-05-16 06:40:00 2025-05-16 07:45:00      1.12019     1.11987    short       3.2      65.0    signal
317 2025-05-16 08:25:00 2025-05-16 09:15:00      1.12069     1.12162    short      -9.3      50.0    signal
318 2025-05-16 09:30:00 2025-05-16 12:15:00      1.12147     1.12021    short      12.6     165.0    signal
